# Module 2 · Session 2 — Lab Notebook
## LLM Internals: Tokenization, Decoding & Sampling — "The Decoding Playground"

**Course:** Generative & Agentic AI Systems
**Time:** ~45 minutes (in-session) · seeds this session's assignment

---

### What you'll do
| Part | What | Time |
|---|---|---|
| 1 | Tokenizer autopsy — tiktoken + GPT-2, weird splits, multilingual costs | 10 min |
| 2 | Distribution surgery — real logits; apply temperature / top-k / top-p yourself | 15 min |
| 3 | Strategy shoot-out — greedy vs beam vs sampling on the same prompt | 15 min |
| 4 | Document behavior patterns — fill the strategy × task matrix | 5 min |

### Requirements
- Python 3.10+ (course standard: 3.12), **CPU only** — GPT-2 small runs fine locally
- Internet on first run (downloads GPT-2, ~500 MB)
- No API keys needed

> **Why GPT-2?** It's small, free, and local — and the tokenization + sampling mechanics are *identical* to frontier models. Expect dated, sometimes goofy text; we're studying the knobs, not the knowledge.

In [ ]:
# Setup — run this cell FIRST (before the break, ideally)
%pip install -q torch transformers tiktoken matplotlib numpy

import numpy as np
import torch
import matplotlib.pyplot as plt
import tiktoken
from transformers import GPT2LMHeadModel, GPT2Tokenizer

torch.manual_seed(42)
np.random.seed(42)

tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Setup OK — GPT-2 loaded,", sum(p.numel() for p in model.parameters())//1_000_000, "M parameters")

---
## Part 1 — Tokenizer Autopsy (10 min)

Models never see text — only token IDs from a fixed vocabulary. Let's dissect how text actually splits.

In [ ]:
def show_tokens(text, tokenizer=tok):
    ids = tokenizer.encode(text)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text!r}  →  {len(ids)} tokens")
    print("  " + " | ".join(repr(p) for p in pieces))
    return ids

show_tokens("The capital of France is Paris.")
show_tokens("strawberry")
show_tokens("How many r's are in strawberry?")
show_tokens("12345 + 12346 = ?")
show_tokens("unbelievably supercalifragilistic")

**Observe:**
- Spaces are *part of the token* (`' Paris'`, not `'Paris'`) — GPT-style byte-level BPE.
- `strawberry` is not 10 letters to the model — it's 2–3 opaque chunks. Now you know why letter-counting is hard for LLMs.
- Numbers split by *frequency luck*, not place value — a root cause of arithmetic inconsistency.

In [ ]:
# tiktoken — the tokenizer used by OpenAI API models; this is what you're billed by
enc = tiktoken.get_encoding("cl100k_base")   # GPT-3.5/4-era encoding

samples = {
    "English":  "The quick brown fox jumps over the lazy dog.",
    "Spanish":  "El rápido zorro marrón salta sobre el perro perezoso.",
    "Arabic":   "الثعلب البني السريع يقفز فوق الكلب الكسول.",
    "Japanese": "素早い茶色の狐がのろまな犬を飛び越える。",
    "Python":   "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)",
}
print(f"{'Language':<10} {'chars':>6} {'tokens':>7} {'tokens/char':>12}")
for name, text in samples.items():
    n = len(enc.encode(text))
    print(f"{name:<10} {len(text):>6} {n:>7} {n/len(text):>12.2f}")

> **Checkpoint 1:** Same meaning, very different token counts. Every extra token is context-window budget *and* API cost. For multilingual products this is a real planning number, not trivia.

**✏️ Exercise 1.1** — Find (a) an English word that splits into 3+ tokens, and (b) two different 5-digit numbers that split differently from each other. Note what this predicts about model behavior on each.

In [ ]:
# ✏️ Exercise 1.1 — your experiments here
show_tokens("...")   # hunt for a 3+ token English word
show_tokens("...")   # compare two 5-digit numbers
# Your predictions as comments:
# ...

---
## Part 2 — Distribution Surgery (15 min)

The model's only output is a **probability distribution over its vocabulary**. Every sampling parameter is surgery on that distribution — and you're about to perform each operation yourself, on real GPT-2 logits.

In [ ]:
PROMPT = "The capital of France is"

with torch.no_grad():
    input_ids = tok.encode(PROMPT, return_tensors="pt")
    logits = model(input_ids).logits[0, -1]     # logits for the NEXT token: shape (50257,)

print("Logit tensor shape:", tuple(logits.shape), "— one raw score per vocab entry")

def top_tokens(probs, k=8):
    vals, idx = torch.topk(probs, k)
    return [(tok.decode([i]), v.item()) for i, v in zip(idx, vals)]

probs = torch.softmax(logits, dim=-1)
print(f"\nTop candidates after '{PROMPT}':")
for t, p in top_tokens(probs):
    print(f"  {t!r:>12}  {p:6.1%}")

### Surgery 1: Temperature — reshape

`softmax(logits / T)` — same ranking, different shape.

In [ ]:
temps = [0.2, 0.7, 1.0, 1.5]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, T in zip(axes, temps):
    p = torch.softmax(logits / T, dim=-1)
    labels, vals = zip(*top_tokens(p, k=6))
    ax.bar(range(6), vals, color="#6C63FF")
    ax.set_xticks(range(6)); ax.set_xticklabels([l.strip() or "␣" for l in labels], rotation=45, fontsize=8)
    ax.set_title(f"T = {T}"); ax.set_ylim(0, 1)
plt.suptitle(f"Same logits, four temperatures — prompt: {PROMPT!r}", y=1.05)
plt.tight_layout(); plt.show()
# 📸 This 4-panel chart is the screenshot Slide 20's placeholder asks for.

### Surgery 2: Top-k — fixed truncation, and Surgery 3: Top-p — adaptive truncation

In [ ]:
def apply_top_k(probs, k):
    vals, idx = torch.topk(probs, k)
    out = torch.zeros_like(probs); out[idx] = vals
    return out / out.sum()

def apply_top_p(probs, p):
    sorted_p, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    keep = cum <= p
    keep[0] = True                       # always keep at least the top token
    out = torch.zeros_like(probs); out[sorted_idx[keep]] = sorted_p[keep]
    return out / out.sum(), int(keep.sum())

p_k = apply_top_k(probs, k=50)
p_p, nucleus_size = apply_top_p(probs, p=0.9)
print(f"top-k (k=50): kept exactly 50 candidates (by definition)")
print(f"top-p (p=0.9): kept {nucleus_size} candidates — ADAPTIVE, tiny because the model is confident here")

# The adaptivity payoff: an open-ended prompt grows the nucleus
open_prompt = "My favorite thing about weekends is"
with torch.no_grad():
    l2 = model(tok.encode(open_prompt, return_tensors="pt")).logits[0, -1]
_, n2 = apply_top_p(torch.softmax(l2, dim=-1), p=0.9)
print(f"\nSame p=0.9 on an open-ended prompt → nucleus size {n2} — the set expands when the model is uncertain")

> **Checkpoint 2:** You just implemented the exact pipeline from the slides: **temperature reshapes → top-k/top-p truncate → sample.** Confident context → nucleus of a few tokens; open-ended context → nucleus of dozens or hundreds. That adaptivity is why top-p became the industry default.

**✏️ Exercise 2.1** — Combine the surgeries: apply `T=1.2` *then* `top_p=0.9` to the "France" logits. Is the nucleus bigger or smaller than at T=1.0? Explain in one comment line why temperature changes the nucleus size even though top-p is 'the same'.

In [ ]:
# ✏️ Exercise 2.1 — your code here
p_hot = torch.softmax(logits / 1.2, dim=-1)
_, n_hot = apply_top_p(p_hot, p=0.9)
print("Nucleus size at T=1.2, p=0.9:", n_hot, " vs at T=1.0:", nucleus_size)
# Why: ...

---
## Part 3 — Strategy Shoot-out (15 min)

Same prompt, every strategy. Watch greedy loop, beam play it safe, and sampling diversify.

In [ ]:
GEN_PROMPT = "In the future, artificial intelligence will"
ids = tok.encode(GEN_PROMPT, return_tensors="pt")

def generate(name, **kwargs):
    out = model.generate(ids, max_new_tokens=40, pad_token_id=tok.eos_token_id, **kwargs)
    text = tok.decode(out[0][ids.shape[1]:])
    print(f"— {name} " + "-"*(60-len(name)))
    print(" ", text.replace("\n", " ").strip(), "\n")

torch.manual_seed(0)
generate("GREEDY (do_sample=False)", do_sample=False)
generate("BEAM SEARCH (num_beams=5)", do_sample=False, num_beams=5)
generate("SAMPLING T=0.7, top_p=0.9", do_sample=True, temperature=0.7, top_p=0.9)
generate("SAMPLING T=1.5 (hot)", do_sample=True, temperature=1.5)
generate("SAMPLING T=0.7 + repetition_penalty=1.3", do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.3)

In [ ]:
# Determinism check — the production-debugging fact of the day
torch.manual_seed(123); out_a = model.generate(ids, max_new_tokens=20, do_sample=True, temperature=0.9, pad_token_id=tok.eos_token_id)
torch.manual_seed(123); out_b = model.generate(ids, max_new_tokens=20, do_sample=True, temperature=0.9, pad_token_id=tok.eos_token_id)
torch.manual_seed(999); out_c = model.generate(ids, max_new_tokens=20, do_sample=True, temperature=0.9, pad_token_id=tok.eos_token_id)
print("Same seed, same settings  → identical:", torch.equal(out_a, out_b))
print("Different seed            → identical:", torch.equal(out_a, out_c))

**✏️ Exercise 3.1** — Run the greedy strategy with `max_new_tokens=80` on the prompt `"The best thing about my job is"`. Does it fall into a repetition loop? At roughly which token does the loop begin? Then fix it *without* enabling sampling (hint: `repetition_penalty`).

**✏️ Exercise 3.2** — Run the T=0.7/top-p=0.9 configuration **5 times** (no manual seed). Rate each output 1–5 for coherence. What's your observed variance — and would you ship this setting for a customer-facing FAQ bot?

In [ ]:
# ✏️ Exercises 3.1 & 3.2 — your experiments here
# ...

---
## Part 4 — Document Behavior Patterns (5 min)

Fill this matrix from YOUR observed evidence (not the slides). It becomes Part A of your assignment.

| Strategy / Config | Observed behavior (your words) | Best-fit task type | Evidence (which cell/run) |
|---|---|---|---|
| Greedy | | | |
| Beam (k=5) | | | |
| T=0.7, top-p=0.9 | | | |
| T=1.5 | | | |
| T=0.7 + rep. penalty | | | |

*(Double-click this cell to edit the table directly.)*

---
## 🏆 Challenges (optional, ranked)

1. **Nucleus tracker (⭐⭐)** — During a 30-token generation at p=0.9, record the nucleus size at every step and plot it. Where does the model get uncertain?
2. **Manual generation loop (⭐⭐)** — Reimplement `generate()` yourself: loop over forward passes, apply your `apply_top_p` from Part 2, sample with `torch.multinomial`, append, repeat. Verify it behaves like the built-in.
3. **Entropy vs quality (⭐⭐⭐)** — For temperatures 0.2 → 2.0 in steps of 0.2, generate 3 samples each and compute mean distribution entropy at each step. Plot entropy vs your subjective coherence ratings — find the cliff where creativity becomes noise.

In [ ]:
# 🏆 Challenge workspace
# Starter for Challenge 2 — the manual loop skeleton:
def my_generate(prompt, n_tokens=30, T=0.8, p=0.9):
    ids_ = tok.encode(prompt, return_tensors="pt")
    for _ in range(n_tokens):
        with torch.no_grad():
            lg = model(ids_).logits[0, -1]
        pr = torch.softmax(lg / T, dim=-1)
        pr, _ = apply_top_p(pr, p)
        nxt = torch.multinomial(pr, 1)
        ids_ = torch.cat([ids_, nxt.view(1, 1)], dim=1)
    return tok.decode(ids_[0])

print(my_generate("In the future, artificial intelligence will"))

---
## Summary

| Concept | What you did |
|---|---|
| Tokenization | Dissected real BPE splits; measured multilingual token inflation with tiktoken |
| Logits → probabilities | Pulled raw GPT-2 logits and softmaxed them into the model's actual next-token beliefs |
| Temperature | Reshaped one distribution four ways and plotted the collapse/flatten spectrum |
| Top-k vs top-p | Implemented both truncations; measured nucleus adaptivity across confident vs open prompts |
| Decoding strategies | Ran greedy/beam/sampling head-to-head; reproduced the greedy repetition loop and fixed it |
| Determinism | Proved seed + settings = reproducibility — the fact behind "why did the same prompt differ?" |

**Next:** the assignment ("The Decoding Playbook") turns your Part 4 matrix into a production-ready parameter guide. See the assignment brief for the rubric.

*Module 2 · Session 2 — Generative & Agentic AI Systems*